In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 1. 라이브러리 설치
!pip install tmdbv3api tqdm

# TMDB 추출  
1. links.csv에서 tmdbId로 TMDB API 서버에 접근  
2. keywords, cast(상위 3명), director 정보를 뽑아서 데이터프레임에 저장  
3. 가중치 텍스트 생성  
- 추천 품질을 높이려면 장르보다는 키워드나 감독이 더 중요하므로 나중에 유사도 계산 시, 키워드가 겹치는 영화끼리의 점수가 장르만 겹치는 영화보다 높게 나오게 하기 위해 가중치를 줌  
- 그냥 액션 영화가 아니라 꿈을 소재로 한 액션 영화를 찾아내기 위한 임베딩  
- 키워드 문자열을 n번 반복해서 이어 붙이는 방식 (아래 코드에서는 3번으로 설정함)

In [6]:
import pandas as pd
import numpy as np
from tmdbv3api import TMDb, Movie
from tqdm import tqdm
import os
import concurrent.futures

# =========================================================
# 🚨 [필수] API 키와 경로 설정
# =========================================================
TMDB_API_KEY = '0abd9b1e6dd17a10596d90cad0a28c0e'
BASE_PATH = '/content/drive/MyDrive/Colab Notebooks/2025-2 빅데이터알고리즘'

# =========================================================
# 1. 기본 데이터 로드
# =========================================================
print("📂 1. MovieLens 데이터 로드...")
try:
    movies = pd.read_csv(os.path.join(BASE_PATH, 'movies.csv'))
    links = pd.read_csv(os.path.join(BASE_PATH, 'links.csv'))
except FileNotFoundError:
    raise FileNotFoundError("movies.csv 또는 links.csv가 없습니다.")

links = links.dropna(subset=['tmdbId'])
links['tmdbId'] = links['tmdbId'].astype(int)

# =========================================================
# 2. TMDB 데이터 고속 수집 (병렬 처리 + 리스트 변환 수정)
# =========================================================
print("🚀 2. TMDB 데이터 수집 시작 (병렬 모드)...")

# 전역 설정
tmdb = TMDb()
tmdb.api_key = TMDB_API_KEY
tmdb.language = 'en'
tmdb.wait_on_rate_limit = True

def fetch_movie_details_safe(tmdb_id):
    # 각 스레드마다 독립 객체 생성
    local_tmdb = TMDb()
    local_tmdb.api_key = TMDB_API_KEY
    local_movie = Movie()

    try:
        # 1. 키워드 호출
        k_res = local_movie.keywords(tmdb_id)
        # [수정] list()로 감싸서 안전하게 변환
        keywords_obj = getattr(k_res, 'keywords', [])
        keywords = [k['name'] for k in list(keywords_obj)]

        # 2. 출연진/감독 호출
        c_res = local_movie.credits(tmdb_id)

        # [수정] list()로 감싸서 슬라이싱 에러 방지 (핵심 수정!)
        cast_obj = getattr(c_res, 'cast', [])
        casts = [c['name'] for c in list(cast_obj)[:3]]

        # 감독 추출
        director = ''
        crew_obj = getattr(c_res, 'crew', [])
        for c in list(crew_obj):
            if c.get('job') == 'Director':
                director = c.get('name')
                break

        return {
            'tmdbId': tmdb_id,
            'keywords': " ".join(keywords),
            'cast': " ".join([str(c).replace(" ", "") for c in casts]),
            'director': str(director).replace(" ", "")
        }
    except Exception as e:
        # 에러 발생 시 메시지 리턴
        return f"ERROR: {e}"

# --- [사전 테스트] ---
print("   👉 [사전 테스트] 첫 번째 영화로 코드 검증 중...")
test_id = int(links['tmdbId'].iloc[0])
test_res = fetch_movie_details_safe(test_id)

if isinstance(test_res, str) and test_res.startswith("ERROR"):
    print(f"\n❌ [테스트 실패] 에러 내용: {test_res}")
    raise ValueError("코드 실행 중 에러가 발생했습니다.")
else:
    print(f"   ✅ 테스트 성공! 데이터 예시: {str(test_res)[:60]}...")

# --- [본 수집 (병렬 처리)] ---
results = []
target_ids = links['tmdbId'].tolist()
print(f"   총 {len(target_ids)}개 영화 정보를 수집합니다. (워커 5명)")

with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
    # 순서대로 결과 받기 (map 사용)
    results_iter = tqdm(executor.map(fetch_movie_details_safe, target_ids), total=len(target_ids))

    for res in results_iter:
        if isinstance(res, dict):
            results.append(res)
        # 에러가 나면 그냥 건너뜀 (너무 많은 에러 로그 방지)

if len(results) == 0:
    raise ValueError("❌ 수집된 데이터가 없습니다.")

print(f"✅ 수집 완료! {len(results)}개 성공.")
tmdb_df = pd.DataFrame(results)

# =========================================================
# 3. 데이터 병합 & Soup 생성 & CSV 저장
# =========================================================
print("🥣 3. 데이터 병합 및 CSV 저장...")

movies_merged = pd.merge(movies, links, on='movieId', how='left')
final_df = pd.merge(movies_merged, tmdb_df, on='tmdbId', how='left')
final_df = final_df.fillna('')

def create_soup(x):
    k = str(x['keywords'])
    c = str(x['cast'])
    d = str(x['director'])
    g = str(x['genres']).replace('|', ' ')
    # 키워드 3배 가중치
    return ( (k + ' ') * 3 + c + ' ' + d + ' ' + g )

final_df['metadata_soup'] = final_df.apply(create_soup, axis=1)

# [핵심] CSV 파일로 저장
save_path = os.path.join(BASE_PATH, 'final_tmdb_data.csv')
final_df.to_csv(save_path, index=False)

print(f"\n🎉 [최종 완료] 파일이 저장되었습니다: {save_path}")
print("   이제 런타임이 끊겨도 이 파일만 불러오면 됩니다!")

📂 1. MovieLens 데이터 로드...
🚀 2. TMDB 데이터 수집 시작 (병렬 모드)...
   👉 [사전 테스트] 첫 번째 영화로 코드 검증 중...
   ✅ 테스트 성공! 데이터 예시: {'tmdbId': 862, 'keywords': 'rescue friendship mission jealo...
   총 9734개 영화 정보를 수집합니다. (워커 5명)


100%|██████████| 9734/9734 [00:14<00:00, 676.39it/s]


✅ 수집 완료! 9622개 성공.
🥣 3. 데이터 병합 및 CSV 저장...

🎉 [최종 완료] 파일이 저장되었습니다: /content/drive/MyDrive/Colab Notebooks/2025-2 빅데이터알고리즘/final_tmdb_data.csv
   이제 런타임이 끊겨도 이 파일만 불러오면 됩니다!


In [9]:
final_df = pd.read_csv(os.path.join(BASE_PATH, 'final_tmdb_data.csv'))
final_df.head(10)

,movieId,title,genres,imdbId,tmdbId,keywords,cast,director,metadata_soup
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709.0,862.0,rescue friendship mission jealousy villain bul...,TomHanks TimAllen DonRickles,JohnLasseter,rescue friendship mission jealousy villain bul...
1,2,Jumanji (1995),Adventure|Children|Fantasy,113497.0,8844.0,giant insect board game disappearance jungle r...,RobinWilliams KirstenDunst BradleyPierce,JoeJohnston,giant insect board game disappearance jungle r...
2,3,Grumpier Old Men (1995),Comedy|Romance,113228.0,15602.0,fishing sequel old man best friend wedding ita...,WalterMatthau JackLemmon Ann-Margret,HowardDeutch,fishing sequel old man best friend wedding ita...
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,114885.0,31357.0,based on novel or book single mother divorce a...,WhitneyHouston AngelaBassett LorettaDevine,ForestWhitaker,based on novel or book single mother divorce a...
4,5,Father of the Bride Part II (1995),Comedy,113041.0,11862.0,daughter baby parent child relationship midlif...,SteveMartin DianeKeaton MartinShort,CharlesShyer,daughter baby parent child relationship midlif...
5,6,Heat (1995),Action|Crime|Thriller,113277.0,949.0,robbery chase obsession detective heist thief ...,AlPacino RobertDeNiro ValKilmer,MichaelMann,robbery chase obsession detective heist thief ...
6,7,Sabrina (1995),Comedy|Romance,114319.0,11860.0,"chauffeur sibling relationship paris, france t...",HarrisonFord JuliaOrmond GregKinnear,SydneyPollack,"chauffeur sibling relationship paris, france t..."
7,8,Tom and Huck (1995),Adventure|Children,112302.0,45325.0,based on novel or book mississippi river male ...,JonathanTaylorThomas BradRenfro EricSchweig,PeterHewitt,based on novel or book mississippi river male ...
8,9,Sudden Death (1995),Action,114576.0,9091.0,explosive hostage ice hockey terrorism vice pr...,Jean-ClaudeVanDamme PowersBoothe RaymondJ.Barry,PeterHyams,explosive hostage ice hockey terrorism vice pr...
9,10,GoldenEye (1995),Action|Adventure|Thriller,113189.0,710.0,computer virus cuba falsely accused secret int...,PierceBrosnan SeanBean IzabellaScorupco,MartinCampbell,computer virus cuba falsely accused secret int...
